In [ ]:
# EXERCISE 4: ADALINE-Based Adaptive Pattern Classification and Prediction
# Dataset: Iris (Setosa = 0, Versicolor = 1)
# Run this entire code in one Google Colab cell.

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# -------------------- LOAD DATASET --------------------

iris = load_iris()
df = pd.DataFrame(iris.data, columns=[
    "Sepal Length", "Sepal Width", "Petal Length", "Petal Width"
])
df["Class"] = iris.target

# Setosa = 0, Versicolor = 1; exclude Virginica
df = df[df["Class"].isin([0, 1])].reset_index(drop=True)

FEATURES = ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]
X = df[FEATURES].values
y = df["Class"].values.astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)


# -------------------- ADALINE MODEL --------------------

class ADALINE:
    def __init__(self, learning_rate=0.01, epochs=100, random_state=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.random_state = random_state
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        rng = np.random.default_rng(self.random_state)
        self.weights = rng.normal(0, 0.01, X.shape[1])
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.epochs):
            # Delta Rule / batch gradient descent
            output = X @ self.weights + self.bias
            errors = y - output

            self.weights += self.learning_rate * (X.T @ errors) / len(y)
            self.bias += self.learning_rate * np.mean(errors)

            updated_output = X @ self.weights + self.bias
            mse = np.mean((y - updated_output) ** 2)
            self.loss_history.append(mse)

        return self

    def predict_output(self, X):
        return np.asarray(X) @ self.weights + self.bias

    def predict(self, X):
        return (self.predict_output(X) >= 0.5).astype(int)

    def accuracy(self, X, y):
        return accuracy_score(y, self.predict(X))

    def mse(self, X, y):
        return mean_squared_error(y, self.predict_output(X))


def train_model(Xtr, ytr, learning_rate=0.01, epochs=100):
    model = ADALINE(
        learning_rate=learning_rate,
        epochs=epochs
    )
    return model.fit(Xtr, ytr)


model = train_model(X_train, y_train)


# -------------------- HELPER FUNCTIONS --------------------

def print_metrics(model, Xte, yte):
    outputs = model.predict_output(Xte)
    predictions = (outputs >= 0.5).astype(int)

    print(f"\nClassification Accuracy: {accuracy_score(yte, predictions) * 100:.2f}%")
    print(f"Final MSE: {mean_squared_error(yte, outputs):.6f}")
    print(f"Misclassified Samples: {np.sum(predictions != yte)}")

    return predictions, outputs


def feature_importance(model, X_sample, feature_names=FEATURES):
    baseline = model.predict_output(X_sample.reshape(1, -1))[0]
    effects = []

    for i, name in enumerate(feature_names):
        modified = X_sample.copy()
        modified[i] += 1.0
        changed_output = model.predict_output(modified.reshape(1, -1))[0]
        change = abs(changed_output - baseline)
        effects.append((name, change))

    effects.sort(key=lambda item: item[1], reverse=True)
    return effects


def noise_experiment(model, Xte, yte):
    rng = np.random.default_rng(42)
    original_predictions = model.predict(Xte)
    original_accuracy = accuracy_score(yte, original_predictions)

    print(f"\nOriginal Accuracy: {original_accuracy * 100:.2f}%")
    results = []

    for level in [5, 10, 20, 30]:
        # Noise is applied to standardized features.
        noise = rng.normal(0, level / 100, Xte.shape)
        noisy_X = Xte + noise
        noisy_predictions = model.predict(noisy_X)
        noisy_accuracy = accuracy_score(yte, noisy_predictions)
        changed = np.sum(noisy_predictions != original_predictions)

        results.append((level, noisy_accuracy, changed))

        print(
            f"{level}% Noise | Accuracy: {noisy_accuracy * 100:.2f}%"
            f" | Changed Predictions: {changed}/{len(yte)}"
        )

    tolerated = [level for level, acc, _ in results
                 if acc >= original_accuracy - 0.05]

    if tolerated:
        print(f"Maximum Noise Level Tolerated (within 5 percentage points): "
              f"{max(tolerated)}%")
    else:
        print("Maximum Noise Level Tolerated: None of the tested levels")

    return results


# -------------------- MENU 1 --------------------

def binary_classification():
    print("\n--- BINARY PATTERN CLASSIFICATION ---")
    outputs = model.predict_output(X_test)
    predictions = (outputs >= 0.5).astype(int)

    result = pd.DataFrame({
        "Actual Class": y_test,
        "Predicted Class": predictions,
        "Linear Output": outputs,
        "Correct": predictions == y_test
    })

    result["Actual Class"] = result["Actual Class"].map(
        {0: "Setosa", 1: "Versicolor"}
    )
    result["Predicted Class"] = result["Predicted Class"].map(
        {0: "Setosa", 1: "Versicolor"}
    )

    print(result.to_string(index=False))
    print(f"\nFinal Classification Accuracy: "
          f"{accuracy_score(y_test, predictions) * 100:.2f}%")
    print(f"Final MSE: {mean_squared_error(y_test, outputs):.6f}")


# -------------------- MENU 2 --------------------

def feature_prediction():
    print("\n--- MULTI-INPUT ADALINE FEATURE PREDICTION ---")
    print("Enter measurements in centimeters.")

    try:
        sample = np.array([
            float(input("Sepal Length: ")),
            float(input("Sepal Width: ")),
            float(input("Petal Length: ")),
            float(input("Petal Width: "))
        ])

        scaled_sample = scaler.transform(sample.reshape(1, -1))
        output = model.predict_output(scaled_sample)[0]
        prediction = int(output >= 0.5)

        print(f"\nLinear Output: {output:.4f}")
        print(f"Predicted Class: "
              f"{'Iris Versicolor' if prediction == 1 else 'Iris Setosa'}")

        print("\nLearned Weights:")
        for name, weight in zip(FEATURES, model.weights):
            print(f"{name}: {weight:.6f}")
        print(f"Bias: {model.bias:.6f}")

    except ValueError:
        print("Invalid input. Please enter numeric measurements.")


# -------------------- MENU 3 --------------------

def feature_influence():
    print("\n--- FEATURE INFLUENCE ANALYZER ---")

    index = int(input(
        f"Select test sample index (0-{len(X_test) - 1}): "
    ))

    if index < 0 or index >= len(X_test):
        print("Invalid sample index.")
        return

    effects = feature_importance(model, X_test[index])

    print("\nFeature Influence Ranking:")
    for rank, (name, effect) in enumerate(effects, start=1):
        print(f"{rank}. {name}: Output change = {effect:.6f}")

    print(f"\nMost Influential Feature: {effects[0][0]}")
    print("Influence is measured by increasing one standardized feature "
          "by one unit while keeping the others unchanged.")


# -------------------- MENU 4 --------------------

def noise_filtering():
    print("\n--- NOISE FILTERING CHALLENGE ---")
    noise_experiment(model, X_test, y_test)


# -------------------- MENU 5 --------------------

def misclassification_analyzer():
    print("\n--- MISCLASSIFICATION PATTERN ANALYZER ---")

    outputs = model.predict_output(X_test)
    predictions = (outputs >= 0.5).astype(int)
    wrong = np.where(predictions != y_test)[0]

    if len(wrong) == 0:
        print("No misclassified test samples.")
        return

    records = []

    for i in wrong:
        original_index = i
        records.append({
            "Test Index": original_index,
            "Sepal Length": X_test_raw[i, 0],
            "Sepal Width": X_test_raw[i, 1],
            "Petal Length": X_test_raw[i, 2],
            "Petal Width": X_test_raw[i, 3],
            "Actual": "Versicolor" if y_test[i] == 1 else "Setosa",
            "Predicted": "Versicolor" if predictions[i] == 1 else "Setosa",
            "Linear Output": outputs[i],
            "Threshold Distance": abs(outputs[i] - 0.5)
        })

    result = pd.DataFrame(records).sort_values("Threshold Distance")
    print(result.to_string(index=False))

    closest = result.iloc[0]
    print("\nSample Closest to Decision Boundary:")
    print(f"Test Index: {int(closest['Test Index'])}")
    print(f"Distance from Threshold: {closest['Threshold Distance']:.6f}")


# -------------------- MENU 6 --------------------

def decision_boundary_analyzer():
    print("\n--- DECISION BOUNDARY ANALYZER ---")
    print("Select two features:")

    for i, name in enumerate(FEATURES):
        print(f"{i + 1}. {name}")

    try:
        first = int(input("First feature number (1-4): ")) - 1
        second = int(input("Second feature number (1-4): ")) - 1

        if first not in range(4) or second not in range(4) or first == second:
            print("Select two different valid features.")
            return

        selected = [first, second]
        selected_names = [FEATURES[first], FEATURES[second]]

        boundary_model = train_model(
            X_train[:, selected], y_train,
            learning_rate=0.01, epochs=200
        )

        print(f"\nSelected Features: {selected_names}")
        print("Decision Boundary Equation:")
        print(
            f"{boundary_model.weights[0]:.6f} * {selected_names[0]} + "
            f"{boundary_model.weights[1]:.6f} * {selected_names[1]} + "
            f"{boundary_model.bias:.6f} = 0.5"
        )

        predictions, outputs = print_metrics(
            boundary_model, X_test[:, selected], y_test
        )

        print("\nPredicted Class Counts:")
        print(f"Setosa (0): {np.sum(predictions == 0)}")
        print(f"Versicolor (1): {np.sum(predictions == 1)}")

        print("\nBoundary Interpretation:")
        print("Output >= 0.5 predicts Versicolor; output < 0.5 predicts Setosa.")

    except ValueError:
        print("Please enter valid numeric choices.")


# -------------------- MENU 7 --------------------

def comparative_analysis():
    print("\n--- COMPARATIVE ANALYSIS ---")

    rows = []
    feature_sets = [
        FEATURES,
        ["Sepal Length", "Sepal Width"],
        ["Petal Length", "Petal Width"]
    ]

    all_combinations = [
        list(pair) for pair in combinations(FEATURES, 2)
    ]
    feature_sets.extend(all_combinations)

    for names in feature_sets:
        indices = [FEATURES.index(name) for name in names]

        comparison_model = train_model(
            X_train[:, indices], y_train,
            learning_rate=0.01, epochs=200
        )

        outputs = comparison_model.predict_output(X_test[:, indices])
        predictions = (outputs >= 0.5).astype(int)
        acc = accuracy_score(y_test, predictions)
        mse = mean_squared_error(y_test, outputs)
        wrong = int(np.sum(predictions != y_test))

        effects = feature_importance(
            comparison_model,
            X_test[0, indices],
            feature_names=names
        )

        noise_results = noise_experiment_quiet(
            comparison_model, X_test[:, indices], y_test
        )

        tolerated = [
            level for level, accuracy, _ in noise_results
            if accuracy >= acc - 0.05
        ]

        rows.append({
            "Model Type": "Multi-Input ADALINE" if len(names) > 1
                          else "Single-Input ADALINE",
            "Selected Features": ", ".join(names),
            "Number of Inputs": len(names),
            "Final MSE": mse,
            "Accuracy (%)": acc * 100,
            "Misclassified Samples": wrong,
            "Most Influential Feature": effects[0][0],
            "Max Tolerated Noise (%)": max(tolerated) if tolerated else 0
        })

    report = pd.DataFrame(rows).sort_values(
        ["Final MSE", "Accuracy (%)"],
        ascending=[True, False]
    )

    print(report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    best = report.iloc[0]
    print("\nBEST FEATURE COMBINATION")
    print(f"Features: {best['Selected Features']}")
    print(f"Final MSE: {best['Final MSE']:.6f}")
    print(f"Accuracy: {best['Accuracy (%)']:.2f}%")
    print("Ranking uses minimum MSE first, then maximum accuracy.")


def noise_experiment_quiet(model, Xte, yte):
    rng = np.random.default_rng(42)
    original_predictions = model.predict(Xte)
    original_accuracy = accuracy_score(yte, original_predictions)
    results = []

    for level in [5, 10, 20, 30]:
        noisy_X = Xte + rng.normal(0, level / 100, Xte.shape)
        predictions = model.predict(noisy_X)
        acc = accuracy_score(yte, predictions)
        changed = int(np.sum(predictions != original_predictions))
        results.append((level, acc, changed))

    return results


# -------------------- MAIN MENU --------------------

def main():
    while True:
        print("\n" + "=" * 55)
        print(" ADALINE IRIS CLASSIFICATION AND PREDICTION SYSTEM")
        print("=" * 55)
        print("1. Binary Pattern Classification")
        print("2. Multi-Input ADALINE Feature Prediction")
        print("3. Feature Influence Analyzer")
        print("4. Noise Filtering Challenge")
        print("5. Misclassification Pattern Analyzer")
        print("6. Decision Boundary Analyzer")
        print("7. Comparative Analysis")
        print("8. Exit")

        choice = input("\nEnter your choice (1-8): ").strip()

        try:
            if choice == "1":
                binary_classification()
            elif choice == "2":
                feature_prediction()
            elif choice == "3":
                feature_influence()
            elif choice == "4":
                noise_filtering()
            elif choice == "5":
                misclassification_analyzer()
            elif choice == "6":
                decision_boundary_analyzer()
            elif choice == "7":
                comparative_analysis()
            elif choice == "8":
                print("Exiting ADALINE System. Goodbye!")
                break
            else:
                print("Invalid choice. Please select 1-8.")
        except Exception as error:
            print(f"An error occurred: {error}")


main()


 ADALINE IRIS CLASSIFICATION AND PREDICTION SYSTEM
1. Binary Pattern Classification
2. Multi-Input ADALINE Feature Prediction
3. Feature Influence Analyzer
4. Noise Filtering Challenge
5. Misclassification Pattern Analyzer
6. Decision Boundary Analyzer
7. Comparative Analysis
8. Exit

Enter your choice (1-8): 8
Exiting ADALINE System. Goodbye!
